In [ ]:
# Cell 1: Install Required Libraries
!pip install tensorflow tensorflow-addons xgboost lightgbm optuna -q

# Cell 2: Imports
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (LSTM, GRU, Dense, Dropout, BatchNormalization,
                                     Attention, MultiHeadAttention, LayerNormalization,
                                     GlobalAveragePooling1D, Conv1D, MaxPooling1D,
                                     Flatten, Add, Concatenate)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (EarlyStopping, ModelCheckpoint, ReduceLROnPlateau,
                                        TensorBoard, CSVLogger)
from sklearn.model_selection import train_test_split
import xgboost as xgb
import joblib
import warnings
warnings.filterwarnings('ignore')

# Cell 3: Load Preprocessed Features
print("Loading features...")
df = pd.read_csv('features_with_labels.csv', index_col=0, parse_dates=True)
print(f"Total features: {len(df.columns)}")
print(f"Data shape: {df.shape}")

# Cell 4: Prepare Data for Training (Classification - Market Analysis)
# Create target variables for analysis
# 1. Future price direction (for trend analysis)
df['future_return_1d'] = df['Close'].shift(-1) / df['Close'] - 1
df['future_return_5d'] = df['Close'].shift(-5) / df['Close'] - 1
df['direction_1d'] = (df['future_return_1d'] > 0).astype(int)
df['direction_5d'] = (df['future_return_5d'] > 0).astype(int)

# 2. Volatility regime prediction (for risk analysis)
df['volatility_next_5d'] = df['Returns'].rolling(5).std().shift(-5)
df['volatility_regime_next'] = pd.qcut(df['volatility_next_5d'], q=3, labels=[0,1,2])

# 3. Price level prediction (support/resistance)
df['touches_resistance'] = ((df['Close'] >= df['Resistance'] * 0.99) & (df['Close'] <= df['Resistance'] * 1.01)).astype(int)

# Drop NaN from shifting
df_clean = df.dropna()
print(f"Shape after target creation: {df_clean.shape}")

# Cell 5: Build 200M Parameter Transformer Model
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim)
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

def create_large_analysis_model(input_shape, num_classes=3):
    """
    Create Transformer model with ~200M parameters for Forex analysis
    """
    inputs = Input(shape=input_shape)
    
    # Initial projection to higher dimension
    x = Dense(1024, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    
    # Positional encoding (simplified)
    pos_encoding = tf.range(start=0, limit=input_shape[0], delta=1)
    pos_encoding = tf.cast(pos_encoding, tf.float32)
    pos_encoding = tf.reshape(pos_encoding, (1, -1, 1))
    pos_encoding = tf.tile(pos_encoding, [1, 1, 1024])
    x = x + pos_encoding
    
    # Multiple Transformer blocks for 200M parameters (12 blocks, large dimensions)
    for i in range(12):
        x = TransformerBlock(embed_dim=1024, num_heads=16, ff_dim=4096)(x)
        x = Dropout(0.1)(x)
    
    # Global pooling and dense layers
    x = GlobalAveragePooling1D()(x)
    
    # Multi-task outputs for comprehensive analysis
    # Task 1: Direction prediction (classification)
    dir_output = Dense(256, activation='relu')(x)
    dir_output = Dropout(0.3)(dir_output)
    dir_output = Dense(128, activation='relu')(dir_output)
    direction = Dense(2, activation='softmax', name='direction')(dir_output)
    
    # Task 2: Volatility regime prediction (classification)
    vol_output = Dense(256, activation='relu')(x)
    vol_output = Dropout(0.3)(vol_output)
    volatility = Dense(3, activation='softmax', name='volatility')(vol_output)
    
    # Task 3: Price level prediction (regression for confidence)
    price_output = Dense(256, activation='relu')(x)
    price_output = Dropout(0.3)(price_output)
    price_level = Dense(1, activation='sigmoid', name='price_level')(price_output)
    
    # Task 4: Trend strength (regression)
    trend_output = Dense(256, activation='relu')(x)
    trend_output = Dropout(0.3)(trend_output)
    trend_strength = Dense(1, activation='linear', name='trend_strength')(trend_output)
    
    model = Model(inputs=inputs, outputs=[direction, volatility, price_level, trend_strength])
    
    # Calculate parameter count
    total_params = model.count_params()
    print(f"Total parameters: {total_params:,} ({total_params/1e6:.1f}M)")
    
    return model

# Cell 6: Prepare Data for Transformer
# Use window of 60 time steps
WINDOW_SIZE = 60
feature_columns = [col for col in df_clean.columns if col not in ['future_return_1d', 'future_return_5d', 
                                                                   'direction_1d', 'direction_5d',
                                                                   'volatility_next_5d', 'volatility_regime_next',
                                                                   'touches_resistance']]

# Create sequences
def create_sequences_for_transformer(data, targets_dict, window_size):
    X, y_dir, y_vol, y_price, y_trend = [], [], [], [], []
    
    for i in range(len(data) - window_size):
        X.append(data[feature_columns].iloc[i:i+window_size].values)
        y_dir.append(targets_dict['direction'][i+window_size])
        y_vol.append(targets_dict['volatility_regime'][i+window_size])
        y_price.append(targets_dict['price_level'][i+window_size])
        y_trend.append(targets_dict['trend_strength'][i+window_size])
    
    return np.array(X), [np.array(y_dir), np.array(y_vol), np.array(y_price), np.array(y_trend)]

targets = {
    'direction': df_clean['direction_1d'],
    'volatility_regime': df_clean['volatility_regime_next'],
    'price_level': df_clean['touches_resistance'],
    'trend_strength': df_clean['ADX'] / 100  # Normalize ADX
}

X, y = create_sequences_for_transformer(df_clean, targets, WINDOW_SIZE)
print(f"X shape: {X.shape}")
print(f"Number of sequences: {len(X)}")

# Cell 7: Train/Validation/Test Split
split1 = int(0.7 * len(X))
split2 = int(0.85 * len(X))

X_train, X_val, X_test = X[:split1], X[split1:split2], X[split2:]
y_train = [y[0][:split1], y[1][:split1], y[2][:split1], y[3][:split1]]
y_val = [y[0][split1:split2], y[1][split1:split2], y[2][split1:split2], y[3][split1:split2]]
y_test = [y[0][split2:], y[1][split2:], y[2][split2:], y[3][split2:]]

print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

# Cell 8: Create and Compile Model
input_shape = (WINDOW_SIZE, len(feature_columns))
model = create_large_analysis_model(input_shape)

# Custom learning rate schedule
initial_learning_rate = 0.0001
lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate, decay_steps=10000, alpha=0.001
)

model.compile(
    optimizer=Adam(learning_rate=lr_schedule, clipnorm=1.0),
    loss={
        'direction': 'sparse_categorical_crossentropy',
        'volatility': 'sparse_categorical_crossentropy',
        'price_level': 'binary_crossentropy',
        'trend_strength': 'mse'
    },
    loss_weights={
        'direction': 0.4,  # More weight to direction
        'volatility': 0.2,
        'price_level': 0.2,
        'trend_strength': 0.2
    },
    metrics={
        'direction': ['accuracy'],
        'volatility': ['accuracy'],
        'price_level': ['accuracy'],
        'trend_strength': ['mae']
    }
)

model.summary()

# Cell 9: Callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),
    ModelCheckpoint('forex_analysis_model_200m.h5', monitor='val_loss', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7),
    CSVLogger('training_log.csv'),
    TensorBoard(log_dir='./logs')
]

# Cell 10: Train Model
print("Starting training...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

# Cell 11: Save Model and Training History
model.save('forex_analysis_complete.h5')
joblib.dump(history.history, 'training_history.pkl')
print("Model saved successfully!")

# Cell 12: Plot Training History
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0,0].plot(history.history['direction_accuracy'], label='Train')
axes[0,0].plot(history.history['val_direction_accuracy'], label='Validation')
axes[0,0].set_title('Direction Prediction Accuracy')
axes[0,0].set_xlabel('Epoch')
axes[0,0].set_ylabel('Accuracy')
axes[0,0].legend()

axes[0,1].plot(history.history['volatility_accuracy'], label='Train')
axes[0,1].plot(history.history['val_volatility_accuracy'], label='Validation')
axes[0,1].set_title('Volatility Regime Accuracy')
axes[0,1].set_xlabel('Epoch')
axes[0,1].set_ylabel('Accuracy')
axes[0,1].legend()

axes[1,0].plot(history.history['loss'], label='Train Loss')
axes[1,0].plot(history.history['val_loss'], label='Validation Loss')
axes[1,0].set_title('Total Loss')
axes[1,0].set_xlabel('Epoch')
axes[1,0].set_ylabel('Loss')
axes[1,0].legend()

axes[1,1].plot(history.history['trend_strength_mae'], label='Train MAE')
axes[1,1].plot(history.history['val_trend_strength_mae'], label='Validation MAE')
axes[1,1].set_title('Trend Strength MAE')
axes[1,1].set_xlabel('Epoch')
axes[1,1].set_ylabel('MAE')
axes[1,1].legend()

plt.tight_layout()
plt.show()

# Print final metrics
print("\n" + "="*50)
print("FINAL MODEL PERFORMANCE")
print("="*50)
print(f"Direction Accuracy: {history.history['val_direction_accuracy'][-1]:.3f}")
print(f"Volatility Accuracy: {history.history['val_volatility_accuracy'][-1]:.3f}")
print(f"Price Level Accuracy: {history.history['val_price_level_accuracy'][-1]:.3f}")
print(f"Trend Strength MAE: {history.history['val_trend_strength_mae'][-1]:.3f}")